In [ ]:
session.sql("DROP TABLE IF EXISTS tmp_medications_stream_snapshot").collect()
print("Temp table dropped")

In [ ]:
session.sql("""
CREATE TEMP TABLE tmp_medications_stream_snapshot AS
SELECT *
FROM STREAM_T_MEDICATIONS
WHERE METADATA$ACTION = 'INSERT'
""").collect()

med_raw = session.table("tmp_medications_stream_snapshot")
med_raw = med_raw.drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID")
print(f"snapshot medications records found")

In [ ]:
ind_columns = ["ROGER_DECISION_DRUG_IND"]

med_clean = med_raw
for c in ind_columns:
    med_clean = med_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("N")
        ).otherwise(trim(col(c)))
    )

code_columns = ["MEDICATION_CLASSIFICATION_CODE"]

for c in code_columns:
    med_clean = med_clean.with_column(
        c,
        upper(trim(col(c)))
    )

desc_columns = ["COMMON_NAME", "MEDICATION_DESC"]

for c in desc_columns:
    med_clean = med_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("NA")
        ).otherwise(trim(col(c)))
    )

user_columns = ["ADD_USER", "MOD_USER"]

for c in user_columns:
    med_clean = med_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("SYSTEM")
        ).otherwise(trim(col(c)))
    )

trim_only_columns = ["SOURCE_FILE_NAME"]

for c in trim_only_columns:
    med_clean = med_clean.with_column(
        c,
        trim(col(c))
    )

med_clean = med_clean.with_column_renamed("LOAD_TIMESTAMP", "RAW_LOAD_TIMESTAMP")
print("med clean")

In [ ]:
valid_med = med_clean.filter(
    col("MED_ID").is_not_null()
)

invalid_med = med_clean.filter(
    col("MED_ID").is_null()
).with_column(
    "ERROR_REASON",
    lit("MED_ID_NULL")
)
print("seperated valid and invalid records")

In [ ]:
checksum_columns = [
    ("MED_ID", "number"),
    ("COMMON_NAME", "string"),
    ("MEDICATION_DESC", "string"),
    ("ROGER_DECISION_DRUG_IND", "string"),
    ("MEDICATION_CLASSIFICATION_CODE", "string"),
    ("ALLOW_FREQUENCY_NUM", "number"),
    ("ADD_PERSON_ID", "number"),
    ("ADD_ORGN_ID", "number"),
    ("MOD_PERSON_ID", "number"),
    ("MOD_ORGN_ID", "number"),
    ("ADD_USER", "string"),
    ("MOD_USER", "string"),
]

checksum_exprs = []
for col_name, col_type in checksum_columns:
    if col_type == "string":
        checksum_exprs.append(coalesce(col(col_name), lit("")))
    else:
        checksum_exprs.append(coalesce(col(col_name).cast("string"), lit("")))

med_final = valid_med.with_column(
    "CHECKSUM",
    sha2(concat_ws(lit("|"), *checksum_exprs), 256)
)

In [ ]:
med_final = med_final.with_column(
    "STAGING_LOADED_TIMESTAMP",
    current_timestamp()
)

med_final.write.save_as_table(
    table_name=f"{DB}.{STG}.{STG_MEDICATIONS}",
    mode="truncate"
)

print(f"MEDICATIONS loaded into {STG}.{STG_MEDICATIONS}")

In [ ]:
invalid_med.create_or_replace_temp_view("tmp_invalid_med")

invalid_count = invalid_med.count()

if invalid_count > 0:
    session.sql(f"""
    INSERT INTO {DB}.{STG}.{INVALID_RECORDS}
    (
        TABLE_NAME,
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        RAW_RECORD
    )
    SELECT
        'T_MEDICATIONS',
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        OBJECT_CONSTRUCT(*)
    FROM tmp_invalid_med
    """).collect()
    print(f"Invalid records saved into {STG}.{INVALID_RECORDS}")
else:
    print("No invalid records")

In [ ]:
rows_processed = med_final.count()
rows_failed = invalid_count

status = 'SUCCESS' if rows_failed == 0 else 'PARTIAL_SUCCESS'

session.call(
    f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    session.sql("SELECT UUID_STRING()").collect()[0][0],
    'STG_MEDICATIONS_LOAD',
    'STAGING',
    datetime.now(),
    datetime.now(),
    rows_processed,
    rows_failed,
    status,
    None
)

session.call(
    f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status,
    'STG_MEDICATIONS_LOAD',
    'STAGING',
    f'MEDICATIONS staging completed. Rows processed: {rows_processed}, Failed rows: {rows_failed}'
)
print("Audit log inserted and email sent")